# Find Best EEG Channels
Quick analysis to find the most important channels for fixation classification.

In [8]:
# Quick channel analysis - find best channels from ALL available channels
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier

# Config  
BASE_DIR = '~/Code/Eye_EEG/data'  # Local path
LABELED_EEG_DATA_DIR = Path(BASE_DIR) / 'data' / 'labeled_eeg_data'
N_SUBJECTS = 6  # Use 3 subjects for speed
N_TOP = 6       # Select top 6 channels

print(f"Finding best {N_TOP} channels from ALL available channels using {N_SUBJECTS} subjects...")

Finding best 6 channels from ALL available channels using 6 subjects...


In [9]:
# Load data, train RF, get best channels (with valid accuracy)
from sklearn.model_selection import train_test_split

N_SUBJECTS = 10  # Use more subjects for better validity

subjects = [d.name for d in LABELED_EEG_DATA_DIR.iterdir() if d.is_dir() and d.name.startswith('P')][:N_SUBJECTS]
all_X, all_y, all_channels = [], [], None

for subj in subjects:
    data_path = LABELED_EEG_DATA_DIR / subj / '500_500_training_data.npz'
    data = np.load(data_path)
    if all_channels is None:
        all_channels = list(data['channel_names'])  # Get ALL available channels
    X = data['X'][:400].astype(np.float32)  # Use 400 trials max per subject
    y = data['y'][:400]
    all_X.append(X)
    all_y.append(y)
    print(f"Loaded {subj}: {X.shape} trials, {len(all_channels)} channels")

# Combine and flatten for RF
X_combined = np.concatenate(all_X, axis=0)
y_combined = np.concatenate(all_y, axis=0)
X_flat = X_combined.reshape(len(X_combined), -1)  # Flatten time*channels

# Train/test split for valid accuracy
X_train, X_test, y_train, y_test = train_test_split(X_flat, y_combined, test_size=0.3, random_state=42, stratify=y_combined)

print(f"\nTraining RF on {X_train.shape} (trials, features)...")
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Get channel importance (sum over time for each channel)
n_channels = len(all_channels)
n_times = X_combined.shape[2]
feature_importance = rf.feature_importances_.reshape(n_channels, n_times)
channel_importance = feature_importance.sum(axis=1)

# Get top channels
top_indices = np.argsort(channel_importance)[::-1][:N_TOP]
top_channels = [all_channels[i] for i in top_indices]

print(f"\nTop {N_TOP} channels (from {len(all_channels)} total):")
for i, (ch, idx) in enumerate(zip(top_channels, top_indices)):
    print(f"{i+1:2d}. {ch:4s} (importance: {channel_importance[idx]:.3f})")

print(f"\nCopy-paste ready list: {top_channels}")
print(f"Train accuracy: {rf.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {rf.score(X_test, y_test):.3f}")

Loaded P19: (400, 32, 500) trials, 32 channels
Loaded P21: (400, 32, 500) trials, 32 channels
Loaded P28: (400, 32, 500) trials, 32 channels
Loaded P17: (400, 32, 500) trials, 32 channels
Loaded P10: (400, 32, 500) trials, 32 channels
Loaded P16: (400, 32, 500) trials, 32 channels
Loaded P20: (400, 32, 500) trials, 32 channels
Loaded P27: (400, 32, 500) trials, 32 channels
Loaded P05: (400, 32, 500) trials, 32 channels
Loaded P34: (400, 32, 500) trials, 32 channels

Training RF on (2800, 16000) (trials, features)...

Top 6 channels (from 32 total):
 1. CP6  (importance: 0.036)
 2. Fz   (importance: 0.035)
 3. P4   (importance: 0.035)
 4. PO4  (importance: 0.034)
 5. F8   (importance: 0.034)
 6. Fp2  (importance: 0.034)

Copy-paste ready list: ['CP6', 'Fz', 'P4', 'PO4', 'F8', 'Fp2']
Train accuracy: 1.000
Test accuracy:  0.527
